In [1]:
from models import *
import torch
from dataloaders import *
import torch.nn as nn
from torch.utils.data import DataLoader
from shenghao_data import *

In [46]:
class MarginToleranceLoss(nn.Module):
    def __init__(self, tol=1, reduction='mean'):
        super().__init__()
        self.tol = tol
        self.reduction = reduction

    def forward(self, pred, y):
        loss = torch.relu(torch.abs(pred - y) - self.tol)
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss  # 'none'

class ToleranceAccuracy(nn.Module):
    def __init__(self, tol=1):
        super().__init__()
        self.tol = tol

    def forward(self, pred, y):
        return (torch.abs(pred - y) <= self.tol).float().mean()

In [36]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [37]:
model = SimpleTransformerModel(d_model=64, n_heads=4, num_layers=4, tropical=True, num_classes=64, activation='relu',
                           tropical_attention_cls = TropicalAttention(64, 4, torch.device('cuda')), pool=True, skip=True).to(device)

In [38]:
ckpt_path = '15_exp/models/FloydWarshallDataset_tropical_0.0001_20000_20260417_041614_relu_best.pth'

state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [39]:
num_train_samples = 50000
num_val_samples = 10000
n = 8
low_train = 1
high_train = 15
low_test = 1
high_test = 15
use_integer = True
num_additional_node = 0
batch_size = 1
shuffle = True

In [40]:
val_dataset = FloydWarshallDataset(num_val_samples, adversarial_range=(10, 20), length_range=(8, 8), noise_prob=0, value_range=(1, 15))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [49]:
loss_criterion = MarginToleranceLoss()#nn.MSELoss()
model.eval()
val_loss = 0.0
val_acc = 0.0
n = 0
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        mask = torch.triu(torch.ones_like(y, dtype=torch.bool), diagonal=1)
        loss = loss_criterion(out[mask], y[mask])
        k = mask.sum().item()
        val_loss += loss.item() * k
        val_acc += metric(out[mask], y[mask]).item() * k
        print(x)
        print(y)
        print(out)
        break
        n += k
val_loss /= n
val_acc /= n

tensor([[[[0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.1429],
          [0.0000, 0.0000, 0.2857],
          [0.0000, 0.0000, 0.4286],
          [0.0699, 0.0000, 0.5714],
          [0.5070, 0.0000, 0.7143],
          [0.4041, 0.0000, 0.8571],
          [0.0000, 0.0000, 1.0000]],

         [[0.0000, 0.1429, 0.0000],
          [0.0000, 0.1429, 0.1429],
          [0.3728, 0.1429, 0.2857],
          [0.0000, 0.1429, 0.4286],
          [0.0000, 0.1429, 0.5714],
          [0.0697, 0.1429, 0.7143],
          [0.2059, 0.1429, 0.8571],
          [0.0897, 0.1429, 1.0000]],

         [[0.0000, 0.2857, 0.0000],
          [0.3728, 0.2857, 0.1429],
          [0.0000, 0.2857, 0.2857],
          [0.0647, 0.2857, 0.4286],
          [0.0000, 0.2857, 0.5714],
          [0.1900, 0.2857, 0.7143],
          [0.3837, 0.2857, 0.8571],
          [0.3396, 0.2857, 1.0000]],

         [[0.0000, 0.4286, 0.0000],
          [0.0000, 0.4286, 0.1429],
          [0.0647, 0.4286, 0.2857],
          [0.0000, 0.4

ZeroDivisionError: float division by zero

In [45]:
print(val_loss, val_acc)

3.0254940378189086 0.10369365707114339
